# CarWorth — Exploratory Data Analysis
**Dataset:** Craigslist Used Cars USA (austinreese/craigslist-carstrucks-data)  
**Goal:** Understand the data distribution, identify patterns, and prepare for cleaning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/raw/vehicles.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Basic Info

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 3. Missing Values

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
cols_with_missing = missing_df[missing_df['Missing Count'] > 0].index
missing_pct[cols_with_missing].sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Column')
plt.tight_layout()
plt.savefig('../assets/missing_values.png', dpi=150)
plt.show()

## 4. Target Variable: Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw price
df['price'].clip(0, 100000).plot(kind='hist', bins=60, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (clipped at $100K)')
axes[0].set_xlabel('Price ($)')

# Log price
log_price = np.log1p(df['price'].clip(lower=1))
log_price.plot(kind='hist', bins=60, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Log(Price+1) Distribution')
axes[1].set_xlabel('log(Price)')

plt.tight_layout()
plt.savefig('../assets/price_distribution.png', dpi=150)
plt.show()

print(f'Price stats:\n{df["price"].describe()}')

## 5. Price Outliers

In [ ]:
print('Price = 0:', (df['price'] == 0).sum())
print('Price < 500:', (df['price'] < 500).sum())
print('Price > 150,000:', (df['price'] > 150000).sum())
print('Price > 1,000,000:', (df['price'] > 1000000).sum())

## 6. Key Categorical Features

In [ ]:
cat_cols = ['manufacturer', 'condition', 'fuel', 'transmission', 'drive', 'type', 'title_status']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts().head(15)
    counts.plot(kind='barh', ax=axes[i], color='steelblue')
    axes[i].set_title(f'{col} (top 15)')
    axes[i].invert_yaxis()

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('../assets/categorical_features.png', dpi=150)
plt.show()

## 7. Numeric Features

In [ ]:
num_cols = ['year', 'odometer']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(num_cols):
    df[col].dropna().plot(kind='hist', bins=50, ax=axes[i], color='teal', edgecolor='white')
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.savefig('../assets/numeric_features.png', dpi=150)
plt.show()

## 8. Price vs Key Features

In [ ]:
df_plot = df[(df['price'] > 500) & (df['price'] < 80000)].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Price vs Year
df_plot.groupby('year')['price'].median().loc[1990:].plot(ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Median Price by Year')
axes[0,0].set_ylabel('Price ($)')

# Price vs Odometer
axes[0,1].scatter(df_plot['odometer'].clip(0, 300000), df_plot['price'], alpha=0.05, s=2, color='coral')
axes[0,1].set_title('Price vs Odometer')
axes[0,1].set_xlabel('Odometer (miles)')
axes[0,1].set_ylabel('Price ($)')

# Price vs Condition
order = ['new', 'like new', 'excellent', 'good', 'fair', 'salvage']
valid_cond = [c for c in order if c in df_plot['condition'].unique()]
df_plot[df_plot['condition'].isin(valid_cond)].boxplot(
    column='price', by='condition', ax=axes[1,0],
    order=valid_cond, showfliers=False
)
axes[1,0].set_title('Price by Condition')
axes[1,0].set_xlabel('')
plt.sca(axes[1,0])
plt.xticks(rotation=30)

# Price vs Fuel
df_plot.boxplot(column='price', by='fuel', ax=axes[1,1], showfliers=False)
axes[1,1].set_title('Price by Fuel Type')
axes[1,1].set_xlabel('')
plt.sca(axes[1,1])
plt.xticks(rotation=30)

plt.suptitle('')
plt.tight_layout()
plt.savefig('../assets/price_vs_features.png', dpi=150)
plt.show()

## 9. Top Manufacturers by Median Price

In [ ]:
top_manufacturers = df_plot.groupby('manufacturer')['price'].agg(['median', 'count'])
top_manufacturers = top_manufacturers[top_manufacturers['count'] > 500].sort_values('median', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 7))
top_manufacturers['median'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Median Price by Manufacturer (min 500 listings)')
ax.set_xlabel('Median Price ($)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../assets/manufacturer_prices.png', dpi=150)
plt.show()

## 10. Geographic Distribution (State)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df['state'].value_counts().plot(kind='bar', ax=ax, color='teal')
ax.set_title('Listings by State')
ax.set_xlabel('State')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../assets/listings_by_state.png', dpi=150)
plt.show()

## 11. Correlation Heatmap (Numeric)

In [ ]:
num_df = df_plot[['price', 'year', 'odometer']].dropna()

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.savefig('../assets/correlation_heatmap.png', dpi=150)
plt.show()

## 12. EDA Summary

| Finding | Detail |
|---|---|
| Shape | ~426K rows, 26 columns |
| Target | `price` — highly right-skewed, use log transform |
| Key features | `year`, `odometer`, `manufacturer`, `condition`, `fuel`, `transmission`, `drive`, `type` |
| Missing cols to drop | `county`, `size` (>70% missing) |
| Price range to keep | $500 – $150,000 |
| Odometer to clip | 0 – 300,000 miles |
| Next step | `02_data_cleaning.ipynb` |